In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_rscf import lps_solver

In [2]:
csv_file = 'closed_shell_atoms_vs_rhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 16 rows found.


In [5]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}

METHOD = "TF0.166666W FA"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.166666
EXC = ['GGA_X_PBE', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 14000
DAMPING = [0.99, 0.9, 0.0001]
D_guess = None
verbose=False

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, D, mu, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "ChemPot,Ha": round(mu, 6),
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Skipping He (Already exists for TF0.166666W FA/UGBS_S)
Skipping Be (Already exists for TF0.166666W FA/UGBS_S)
Skipping Ne (Already exists for TF0.166666W FA/UGBS_S)
Skipping Mg (Already exists for TF0.166666W FA/UGBS_S)
Skipping Ar (Already exists for TF0.166666W FA/UGBS_S)
Skipping Ca (Already exists for TF0.166666W FA/UGBS_S)
Calculating Zn with TF0.166666W FA...
Calculated Energy: -1773.7277 Hartree
Calculating Kr with TF0.166666W FA...
Calculated Energy: -2741.6652 Hartree


In [6]:
df

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha","ChemPot,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFD0.166666W,He,UGBS_S,6,1000,-2.951247,-0.067106,185,True,0.90,0.9,0.0010
1,TFD0.166666W,Be,UGBS_S,6,1000,-15.040391,-0.069849,233,True,0.90,0.9,0.0010
2,TFD0.166666W,Ne,UGBS_S,6,1000,-132.508856,-0.072529,827,True,0.90,0.9,0.0010
3,TFD0.166666W,Mg,UGBS_S,6,1000,-204.540690,-0.072963,1732,True,0.90,0.9,0.0010
4,TFD0.166666W,Ar,UGBS_S,6,1000,-537.208760,-0.073833,2513,True,0.90,0.9,0.0010
5,TFD0.166666W,Ca,UGBS_S,6,1000,-690.396266,-0.074040,3724,True,0.90,0.9,0.0010
6,TFD0.166666W,Zn,UGBS_S,6,1000,-1812.192490,-0.074767,7974,True,0.99,0.9,0.0001
7,TFD0.166666W,Kr,UGBS_S,6,1000,-2795.914635,-0.075063,8692,True,0.99,0.9,0.0001
8,TF0.166666W PBEx,He,UGBS_S,6,1000,-3.097069,-0.081193,238,True,0.90,0.9,0.0010
9,TF0.166666W PBEx,Be,UGBS_S,6,1000,-15.388877,-0.081756,348,True,0.90,0.9,0.0010


In [7]:
df.to_csv(csv_file, index=False)